# Transform Customer Data
1. Remove records with NULL customer_id
2. Remove extract duplicate records
3. Remove duplicate records based on created_timestamp
4. CAST the columns to the correct Data Type
5. Write transformed data to the Silver schema

In [0]:
df = spark.sql("""
SELECT * 
    FROM gizmobox.bronze.py_customers               
""")

display(df)


In [0]:
## other method
df = spark.table('gizmobox.bronze.py_customers')
display(df) 

In [0]:
## we can use options with this method
df = spark.read.table('gizmobox.bronze.py_customers')
display(df)

## 1. Remove records with NULL customer_id

In [0]:
df = spark.table('gizmobox.bronze.py_customers')

'''
Method using sql syntax
'''
# df_filtered = df.filter('customer_id is not null')

'''
Method using python method isNotNull()
'''
# df_filtered = df.filter(df.customer_id.isNotNull())

'''
Using df.where
'''
df_filtered = df.where(df.customer_id.isNotNull())

display(df_filtered)


## 2. Remove exact duplicate records

[DataFrame.distinct()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.distinct.html?highlight=distinct#pyspark.sql.DataFrame.distinct) - Returns a Df containing the distinct rows of this df

[DataFrame.dropDuplicates()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropDuplicates.html#pyspark.sql.DataFrame.dropDuplicates) - Return a new DataFrame with duplicate rows removed, optionally only considering certain columns.

In [0]:
%python

'''
Using DataFrame.distinct()
'''
# df_distinct = df_filtered.distinct()
'''
Using DataFrame.dropDuplicates()
'''
df_distinct = df_filtered.dropDuplicates()

display(df_distinct)

#### 3. Remove duplicate records based on created_timestamp


In [0]:
from pyspark.sql import functions as F



df_max_ts = df_distinct.groupBy('customer_id') \
    .agg(F.max('created_timestamp').alias('max_created_timestamp'))    

display(df_max_ts)

[Select Method](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.select.html?highlight=select#pyspark.sql.DataFrame.select)

In [0]:
df_distinct_customer = (
    df_distinct
    .join(df_max_ts, (df_distinct.customer_id == df_max_ts.customer_id) &
                     (df_distinct.created_timestamp == df_max_ts.max_created_timestamp),
                     'inner')
                    .select(
                        df_distinct['*']
                    )
)
display(df_distinct_customer)

#### 4. Cast the column values to the correct data type



In [0]:
df_casted_customer = (
    df_distinct_customer
    .select(df_distinct_customer.created_timestamp.cast('timestamp'),
            df_distinct_customer.customer_id,
            df_distinct_customer.customer_name,
            df_distinct_customer.date_of_birth.cast('date'),
            df_distinct_customer.email,
            df_distinct_customer.member_since.cast('date'),
            df_distinct_customer.telephone)
)
display(df_casted_customer)

##### 5. Write Data to a Delta Table

In [0]:
df_casted_customer.writeTo('gizmobox.silver.py_customers').createOrReplace()

In [0]:
%sql
Select * from gizmobox.silver.py_customers